# 23. Series与DataFrame

<!-- module-learning-arc:start -->
> **Pandas 模块主线｜第 2 / 10 步：读懂并定位表中信息**
>
> **持续应用背景：** 搭建电商履约异常追踪台：把订单、客户、商品和履约信息整理成安全合并的事实表，再生成趋势指标和异常工单。
>
> **承接上一阶段：** Pandas 模块入门  →  **本章任务：** Series与DataFrame  →  **下一步：** 选择、筛选与排序
>
> **大作业连接：** 本章练习将成为《电商履约异常追踪台》的一部分，最终需要从多表质量审计走到订单粒度事实表、窗口趋势和可复核异常工单。
<!-- module-learning-arc:end -->


## 本章场景

数据分析和清洗的第一步，就是把散落在 Excel、CSV 或数据库里的数据，装进 Python 能“认得到”的结构里。



## 本章目标

学完本章，你将能够：

- **理解**：理解 Series/DataFrame 的轴、索引对齐与数据类型。
- **操作**：能创建 Series/DataFrame 并做取值、索引、聚合。
- **迁移**：能读懂一张经营表的行/列意义并取出、汇总所需部分。


## 23.1 核心概念

**背景引入**：数据分析和清洗的第一步，就是把散落在 Excel、CSV 或数据库里的数据，装进 Python 能“认得到”的结构里。Series 像一列带标签的数，DataFrame 像一张带列名的表——它们正是 Pandas 处理真实数据的“零件箱”。学完这一章，你就能把原始数据读进来、按标签取数、做对齐合并，后面的分组汇总也就有了落脚点。

- Series是一维带标签数组，DataFrame是二维表格。
- Pandas运算优先按索引标签对齐，而不是仅按位置。
- 数据类型决定可用操作和缺失值表示。

> **直观类比**：Pandas 的索引就像名册上的“学号”：两表相加是按“学号”对号，而不是按“排第几个”硬对；学号对不上的同学，合并表里就在他那格留一个“空缺”（NaN），提醒你“缺人”不等于“0 人”，确定要补成 0 再用 fill_value=0。


## 23.2 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| Series() | `pd.Series()`、`sales['华南']` | Series同时保存值和索引，索引可以表达业务标签。 | 把默认整数索引当成稳定业务主键 |
| DataFrame() | `pd.DataFrame()` | DataFrame的每个列名对应一列同长度数据。 | 忽略索引对齐产生的缺失值 |
| shape、columns 和 dtypes | `pd.DataFrame()`、`data.columns.tolist()` | 先检查表格形状、列名和数据类型，再开始分析。 | 混合类型导致整列变成object |
| Series 对齐 | `pd.Series()` | Pandas按索引标签对齐，而不是只按位置相减。 | 把默认整数索引当成稳定业务主键 |
| head()、tail() 和 sample() | `pd.DataFrame()`、`data.head()`、`data.tail()`、`data.sample()` | 抽查表格时同时看开头、结尾和随机样本。 | 忽略索引对齐产生的缺失值 |


## 23.3 示例 1：创建Series

**背景引入**：三月份各地区的销售额 120、150、180 摆在手上，直接 `[120, 150, 180]` 能存，但打印出来是一串没名头的数字，想查“华南卖了 150”得靠数位置。给数组配上**业务标签**当索引，每个数字是谁的、该怎么查就一目了然。

**讲解**：`pd.Series` 把一批数值和它们的标签打包成一个整体，索引就是每个值旁边的“铭牌”。

- `pd.Series([120,150,180], index=["华东","华南","华北"])`：值在前、索引在 `index`，一一对应；
- `name="sales"` 给这一列起个名字，打印时更清楚它代表销售额；
- `sales["华南"]` 用标签直接取“华南”的值，不用数第几个；`sales.sum()` 对整列求和，一步算完；
- **口诀**：Series 是一列带名片的数，索引就是名片上写的“是谁”。


In [ ]:
import pandas as pd

sales = pd.Series(
    [120, 150, 180], index=["华东", "华南", "华北"], name="sales"
)
print(sales)
print("华南:", sales["华南"])
print("合计:", sales.sum())


## 23.4 示例 2：创建DataFrame

**背景引入**：光有几笔金额还不够，你手里是订单明细——订单号、地区、金额、是否已付款混在一起，列数一多，Series 装不下。这时需要一张**既有行又有列名**的表格，把每个字段各归各列。

**讲解**：`pd.DataFrame` 用字典来建表，字典的键就是列名，每个键对应的列表就是这一列的全部数据。

- `{"order_id": ["A1",...], "region": [...], ...}`：每个键是一列，值是一个同长度列表；
- 各列长度**必须一致**，否则会报错——因为一张表里每一“拍”都要对齐；
- `orders.dtypes` 检查每列类型：`amount` 是 float（带小数）、`paid` 是 bool，混在一起毫无压力；
- **口诀**：字典键就是列头，一行一本“数据账”，各列长度要对齐。


In [ ]:
orders = pd.DataFrame(
    {
        "order_id": ["A1", "A2", "A3", "A4"],
        "region": ["华东", "华南", "华东", "华北"],
        "amount": [320.0, 880.0, 460.0, 1250.0],
        "paid": [True, True, False, True],
    }
)
print(orders)
print(orders.dtypes)


## 23.5 示例 3：索引对齐

**背景引入**：一月和三月的销售表，华东、华南、华北三个地区都记了，可二月突然多了个“西南”。直接拿二月减一月，两边都对不上的地区会发生什么？这正是真实报表里最常见的“两张表标签对不上”的坑。

**讲解**：Pandas 做主相减时**按索引标签对齐**，不是傻傻地按位置减；对不上的位置就留成 `NaN`。

- `february - january`：标签相同的“华东、华南”相减，只有一边有值的“华北、西南”变成 `NaN`；
- `NaN` 表示“这里缺数据”，不会悄悄算错，而是在提醒你——这正是它贴心的设计；
- `january.add(february, fill_value=0)`：用 `fill_value=0` 把缺失当 0 补上再相加，就能得到一个完整合计；
- **口诀**：Pandas 相减不看位置看标签，对不上的就亮“NaN”黄灯，fill_value=0 补零再算。


In [ ]:
january = pd.Series({"华东": 120, "华南": 98, "华北": 110})
february = pd.Series({"华东": 150, "华南": 132, "西南": 86})
growth = february - january
print(growth)
print("填零后合计:\n", january.add(february, fill_value=0))


## 23.6 核心操作独立示例

下面每个代码单元格只演示一个核心方法、函数或语法操作。请先阅读方法名称和任务说明，再单独运行当前单元格；示例尽量自带最小输入，不要求依赖前一个单元格留下的变量。


In [ ]:
# Series()
# Series同时保存值和索引，索引可以表达业务标签。
import pandas as pd

sales = pd.Series(
    [120, 150, 180], index=["华东", "华南", "华北"], name="sales"
)
print(sales)
print("华南:", sales["华南"])


In [ ]:
# DataFrame()
# DataFrame的每个列名对应一列同长度数据。
import pandas as pd

orders = pd.DataFrame({"order_id": ["A1", "A2"], "amount": [320, 880]})
print(orders)


In [ ]:
# shape、columns 和 dtypes
# 先检查表格形状、列名和数据类型，再开始分析。
import pandas as pd

data = pd.DataFrame({"name": ["A", "B", "C"], "score": [82, 91, 76]})
print("shape:", data.shape)
print("columns:", data.columns.tolist())
print("dtypes:\n", data.dtypes)


In [ ]:
# Series 对齐
# Pandas按索引标签对齐，而不是只按位置相减。
import pandas as pd

east = pd.Series({"一月": 120, "二月": 150})
south = pd.Series({"二月": 98, "三月": 132})
print(east + south)


In [ ]:
# head()、tail() 和 sample()
# 抽查表格时同时看开头、结尾和随机样本。
import pandas as pd

data = pd.DataFrame({"id": range(1, 6), "value": [10, 20, 30, 40, 50]})
print(data.head(2))
print(data.tail(2))
print(data.sample(2, random_state=1))



**练一练 19.6**：给你华东、华南、华北三个地区的销售额 120、150、180。完成三件事：① 用 Series 把数据存起来，索引用地区名；② 按标签取出“华南”的值；③ 再建一个 Series bonus（华东 10、华南 20、西南 30），让它与 sales 相加，观察索引对齐后产生的 NaN。数据用简单数值即可。


In [ ]:
# 请在下方填写代码
import pandas as pd

# TODO ①: 创建 Series sales，数据 [120, 150, 180]，索引 ["华东", "华南", "华北"]

# TODO ②: 取出“华南”的值并打印

bonus = pd.Series({"华东": 10, "华南": 20, "西南": 30})

# TODO ③: 把 sales 与 bonus 相加（按索引对齐），结果保存在 aligned 并打印


In [ ]:
import pandas as pd

# ① 创建 Series，数据与索引用列表传入
sales = pd.Series([120, 150, 180], index=["华东", "华南", "华北"])
# ② 按标签取“华南”
south = sales["华南"]
print("华南:", south)

bonus = pd.Series({"华东": 10, "华南": 20, "西南": 30})
# ③ 相加时 Pandas 按索引标签对齐，出现“西南”等缺失 -> NaN
aligned = sales + bonus
print(aligned)


## 23.7 公开大型数据实战

下面使用 UCI Machine Learning Repository 的 Online Retail 公开数据集。原始数据包含 541,909 条英国在线零售交易，本课程使用固定随机种子抽取的 200,000 行子集。分析时在完整子集上计算，只展示摘要或少量样本。


In [ ]:
import numpy as np
import pandas as pd

# UCI Machine Learning Repository: Online Retail
# 原始数据 541,909 行；课程使用固定随机种子抽取的 200,000 行子集。
data_url = "/datasets/uci_online_retail_200k.csv"
large_orders = pd.read_csv(
    data_url,
    parse_dates=["InvoiceDate"],
    dtype={
        "InvoiceNo": "string",
        "StockCode": "string",
        "Description": "string",
        "Country": "category",
    },
).rename(
    columns={
        "InvoiceNo": "order_id",
        "StockCode": "stock_code",
        "Description": "description",
        "Quantity": "quantity",
        "InvoiceDate": "order_time",
        "UnitPrice": "unit_price",
        "CustomerID": "customer_id",
        "Country": "country",
    }
)
large_orders["sales"] = (
    large_orders["quantity"] * large_orders["unit_price"]
).round(2)
large_orders["status"] = np.where(
    large_orders["order_id"].str.startswith("C")
    | (large_orders["quantity"] < 0),
    "取消/退货",
    "完成",
)
print("UCI Online Retail 公开数据：")
print(f"  {len(large_orders):,} 行 × {large_orders.shape[1]} 列")
print(
    "内存占用：",
    f"{large_orders.memory_usage(deep=True).sum() / 1024**2:.1f} MB",
)
large_orders.head()


In [ ]:
print(large_orders.info(memory_usage="deep"))
print("\n数值列概览：")
display(large_orders[["quantity", "unit_price", "sales"]].describe().round(2))
print("唯一订单数：", large_orders["order_id"].nunique())


## 23.8 独立迁移练习

替换一个字段或分组口径，并核对处理前后的行数与粒度。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# TODO: 在此粘贴或改写最接近的示例。
# 记录：我改了什么？预期会发生什么？实际观察到什么？
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print({"修改": change_note, "预期": expected_change, "观察": observed_change})


## 23.9 本章实训：分组汇总与粒度

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd

orders = pd.DataFrame(
    {
        "region": ["华东", "华东", "华南", "华南"],
        "channel": ["线上", "线下", "线上", "线下"],
        "sales": [120, 80, 150, 100],
    }
)
summary = orders.groupby("region", as_index=False)["sales"].sum()
print(summary)
print("汇总表每一行代表一个地区")


### 23.9.1 第一个结果怎么读

先确认明细表一行代表一笔订单，再确认汇总表一行代表一个地区。`groupby` 的字段决定结果的粒度。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。



In [ ]:
orders["sales_level"] = orders["sales"].map(
    lambda value: "高" if value >= 120 else "普通"
)
print(orders)
print(orders["sales_level"].value_counts())


### 23.9.2 第二个结果怎么读

第二个实验只增加一个分类列，不改变原始销售额。练习解释：什么时候应该新增列，什么时候应该直接筛选行？

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。



## 23.10 错误恢复：脏数据转换怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

raw = pd.Series(["12", "unknown", "18", ""])
converted = pd.to_numeric(raw, errors="coerce")
print("转换结果：")
print(converted)
print("无法转换的数量：", converted.isna().sum())
print("后续可以选择删除、填充或回查原始值。")


### 23.10.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

errors="coerce" 会把无法转换的值记录为缺失，适合先完成质量盘点；不要在没有统计数量前直接删除。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。



## 23.11 易错点提醒

- 把默认整数索引当成稳定业务主键
- 忽略索引对齐产生的缺失值
- 混合类型导致整列变成object


## 23.12 练习与作业

1. 创建商品DataFrame
2. 包含名称、价格和库存
3. 计算库存总价值列

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 23.13 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“创建商品DataFrame”。
2. **独立完成**：不复制示例代码，完成“包含名称、价格和库存”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“计算库存总价值列”，用一两句话说明你修改了什么。

### 23.13.1 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 23.13.2 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


In [ ]:
import pandas as pd

# TODO: 计算库存总价值列
# TODO：请在下方完成 —— 19.13 练习与作业 1. 创建商品DataFrame 2. 包含名称、价格和库存 3. 计算库存总价值列 提交前检查


In [ ]:
import pandas as pd

products = pd.DataFrame(
    {
        "name": ["键盘", "鼠标", "耳机"],
        "price": [299.0, 129.0, 499.0],
        "stock": [18, 32, 12],
    }
)
products["stock_value"] = products["price"] * products["stock"]
print(products)
print("库存总价值:", products["stock_value"].sum())


## 23.14 小结

理解Series与DataFrame的索引对齐、列类型和基础运算。

**迁移思考**：

1. 如果两个 Series 的索引完全不同，相加后会得到什么结果？如何处理？
2. 为什么 Pandas 要按索引标签对齐而不是按位置对齐？这种设计解决了什么问题？



### 23.14.1 你已经掌握

- 创建Series和DataFrame
- 读取索引与列
- 检查数据类型
- 理解按标签自动对齐



### 23.14.2 验收标准

- 输入、计算和输出单元格完整。
- 关键变量类型、形状或数值可核对。
- 结论引用输出证据，并注明适用范围。



### 23.14.3 需要注意

- 把默认整数索引当成稳定业务主键
- 忽略索引对齐产生的缺失值
- 混合类型导致整列变成object



### 23.14.4 完成检查

- [ ] 能够创建Series和DataFrame
- [ ] 能够读取索引与列
- [ ] 能够检查数据类型
- [ ] 能够理解按标签自动对齐



### 23.14.5 排错顺序

1. 从上到下重新运行依赖单元格。
2. 检查变量类型、列名、形状和缺失值。
3. 缩小输入范围，定位产生错误的最小步骤。
4. 修复后重新运行完整流程。

